In [9]:
import gzip
import json
import os
import pickle

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [10]:
train_df = pd.read_csv(
    "../files/input/train_data.csv.zip",
    index_col=False,
    compression="zip",
)
test_df = pd.read_csv(
    "../files/input/test_data.csv.zip",
    index_col=False,
    compression="zip",
)

train_df.head()

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,jazz,2016,7.40,8.500,15059,Petrol,Dealer,Automatic,0
1,i10,2013,4.00,4.600,30000,Petrol,Dealer,Manual,0
2,TVS Apache RTR 180,2011,0.50,0.826,6000,Petrol,Individual,Manual,0
3,eon,2016,3.15,4.430,15000,Petrol,Dealer,Manual,0
4,Royal Enfield Thunder 350,2013,1.25,1.500,15000,Petrol,Individual,Manual,0


In [11]:
def clean(df):
    df = df.copy()
    df["Age"] = 2021 - df["Year"]
    df = df.drop(columns=["Year", "Car_Name"])
    return df


train_df = clean(train_df)
test_df = clean(test_df)

train_df.head()

,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Age
0,7.40,8.500,15059,Petrol,Dealer,Automatic,0,5
1,4.00,4.600,30000,Petrol,Dealer,Manual,0,8
2,0.50,0.826,6000,Petrol,Individual,Manual,0,10
3,3.15,4.430,15000,Petrol,Dealer,Manual,0,5
4,1.25,1.500,15000,Petrol,Individual,Manual,0,8


In [12]:
target = "Present_Price"

x_train = train_df.drop(columns=[target])
y_train = train_df[target]

x_test = test_df.drop(columns=[target])
y_test = test_df[target]

x_train.head()

,Selling_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Age
0,7.40,15059,Petrol,Dealer,Automatic,0,5
1,4.00,30000,Petrol,Dealer,Manual,0,8
2,0.50,6000,Petrol,Individual,Manual,0,10
3,3.15,15000,Petrol,Dealer,Manual,0,5
4,1.25,15000,Petrol,Individual,Manual,0,8


In [13]:
categorical = ["Fuel_Type", "Selling_type", "Transmission", "Owner"]
numerical = [c for c in x_train.columns if c not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", MinMaxScaler(), numerical),
    ],
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("select_k_best", SelectKBest(score_func=f_regression)),
        ("regressor", LinearRegression()),
    ],
)

pipeline

,steps,"[('preprocessor', ...), ('select_k_best', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
param_grid = {
    "select_k_best__k": range(1, 14),
}

model = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)

model.fit(x_train, y_train)

print("Mejores parámetros:", model.best_params_)

Mejores parámetros: {'select_k_best__k': 13}


In [15]:
os.makedirs("../files/models", exist_ok=True)

with gzip.open("../files/models/model.pkl.gz", "wb") as file:
    pickle.dump(model, file)

print("Modelo guardado en files/models/model.pkl.gz")

Modelo guardado en files/models/model.pkl.gz


In [16]:
def compute_metrics(dataset, x, y):
    y_pred = model.predict(x)
    return {
        "type": "metrics",
        "dataset": dataset,
        "r2": round(r2_score(y, y_pred), 3),
        "mse": round(mean_squared_error(y, y_pred), 3),
        "mad": round(mean_absolute_error(y, y_pred), 3),
    }


metrics = [
    compute_metrics("train", x_train, y_train),
    compute_metrics("test", x_test, y_test),
]

os.makedirs("../files/output", exist_ok=True)
with open("../files/output/metrics.json", "w", encoding="utf-8") as file:
    for row in metrics:
        file.write(json.dumps(row) + "\n")

metrics

[{'type': 'metrics',
  'dataset': 'train',
  'r2': 0.898,
  'mse': 5.556,
  'mad': 1.541},
 {'type': 'metrics',
  'dataset': 'test',
  'r2': 0.748,
  'mse': 30.734,
  'mad': 2.362}]